# 9 Ensamblador

### 9.9 Objetivo

Ensamblar modelos YA construidos
Los modelos a ensamblar deben ser de la misma Modalidad
 * Este script está armado para Analista Sr
 * Este script corre unicamente en Google Cloud

## 9.9.1  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [25]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Aug 24 23:28:25 2025"

In [26]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,724174,38.7,1435128,76.7,1435128,76.7
Vcells,1374162,10.5,15395680,117.5,18244271,139.2


In [27]:
require("data.table")

Loading required package: data.table



#### Parametros

Se debe poner en PARAM$modelos  la lista de modelos que se va a ensamblar, y deben ser por lo menos DOS

In [29]:
PARAM <- list()
PARAM$modelos <- c( "WF950106", "WF950107", "WF950108","WF950109","WF950111","WF950110")

PARAM$experimento <- 999008

#### Carpeta del Experimento

In [30]:
# carpeta de trabajo

setwd("~/buckets/b1/exp")
experimento_folder <- paste0("EN", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("~/buckets/b1/exp/", experimento_folder ))

### 9.9.2   Ensamblado

In [31]:
# lectura del primer  predicion.txt
tb_prediccion  <-  fread( paste0( "~/buckets/b1/exp/", PARAM$modelos[1], "/prediccion.txt" ) )
tb_prediccion <- tb_prediccion[,c("numero_de_cliente", "prob"), with=FALSE]
setorder(tb_prediccion, numero_de_cliente)

In [32]:
buenos <- 1 
for( imodelo in seq(2, length(PARAM$modelos)) )
{
  tbl <- fread( paste0( "~/buckets/b1/exp/", PARAM$modelos[imodelo], "/prediccion.txt" ) )
  if( nrow(tbl) == nrow( tb_prediccion ) )
    {
      setorder(tbl, numero_de_cliente)
      tb_prediccion[, prob := prob + tbl$prob ]
      buenos <- buenos +1 
    }
}

# fui sumando, ahora divido por la cantidad para tener el promedio
tb_prediccion[, prob := prob / buenos ]

# grabo a disto, esto SI PUEDE ser reutilizado
fwrite( tb_prediccion,
  file = "prediccion.txt",
  sep= "\t"
)

In [33]:
tb_prediccion

numero_de_cliente,prob
<int>,<dbl>
29183733,0.0006670274
29184468,0.0005206846
29185245,0.0155829768
29186441,0.0012281139
29186475,0.0012047245
29187730,0.0005224607
29187764,0.0007587509
29187961,0.0015736938
29189899,0.0012704589


### 9.9.3 Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggles

In [9]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

PARAM$kaggle$competencia <- "data-mining-analista-sr-2025-a"
PARAM$kaggle$cortes <- seq(9000, 12000, by = 500)

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0(
    "./kaggle/EN",
    PARAM$experimento, "_",
    envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios, "  Ensemble", "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
}

Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit -c data-mining-analista-sr-2025-a -f ./kaggle/EN999008_9000.csv -m 'envios=9000  Ensemble'' had status 1”


In [10]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Aug 24 23:24:57 2025"